# 03 — Predict no-shows

## Task 3.1 

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

df = pd.read_csv("appointments_clean.csv")
X = df[["age", "distance_km", "fee_inr", "lead_days", "prior_no_shows", "reminder_sent", "booking_channel", "visit_type"]].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"], dtype=int)
y = df["no_show"]
print("X columns:", X.shape[1], list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train:", len(X_train), "Test:", len(X_test))
print("No-shows in test:", y_test.sum())
print("Baseline accuracy (always Attended):", round((y_test == 0).mean(), 4), "=", (y_test == 0).sum(), "of", len(y_test))

X columns: 11 ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']
Train: 792 Test: 198
No-shows in test: 49
Baseline accuracy (always Attended): 0.7525 = 149 of 198


## Task 3.2

In [4]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)
pred = lr.predict(X_test_s)
prob = lr.predict_proba(X_test_s)[:, 1]
lr_acc, lr_rec, lr_auc = accuracy_score(y_test, pred), recall_score(y_test, pred), roc_auc_score(y_test, prob)
print("Accuracy:", round(lr_acc, 3))
print("Confusion matrix:\n", confusion_matrix(y_test, pred))
print("Precision:", round(precision_score(y_test, pred), 3), "Recall:", round(lr_rec, 3),
      "F1:", round(f1_score(y_test, pred), 3), "ROC-AUC:", round(lr_auc, 3))

for name, coef in zip(X.columns, lr.coef_[0]):
    print(f"{name:28s} {coef:+.3f}")

i = list(X.columns).index("reminder_sent")
print("exp(coef) for reminder_sent (per 1 SD of scaled feature):", round(np.exp(lr.coef_[0][i]), 2))
print("scale_ (SD) of reminder_sent:", round(scaler.scale_[i], 2))
print("exp(coef / scale) for reminder No -> Yes:", round(np.exp(lr.coef_[0][i] / scaler.scale_[i]), 2))

Accuracy: 0.788
Confusion matrix:
 [[139  10]
 [ 32  17]]
Precision: 0.63 Recall: 0.347 F1: 0.447 ROC-AUC: 0.789
age                          -0.254
distance_km                  +0.253
fee_inr                      +0.453
lead_days                    +0.614
prior_no_shows               +0.650
reminder_sent                -0.581
booking_channel_App          -0.047
booking_channel_Phone        +0.216
booking_channel_Walk-In      -0.186
visit_type_Consultation      +0.016
visit_type_Follow-up         -0.016
exp(coef) for reminder_sent (per 1 SD of scaled feature): 0.56
scale_ (SD) of reminder_sent: 0.49
exp(coef / scale) for reminder No -> Yes: 0.31


## Task 3.3 

In [5]:
tree_full = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
tree3 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
for name, t in [("Full-depth tree", tree_full), ("Depth-3 tree", tree3)]:
    print(name, "| train acc:", round(t.score(X_train, y_train), 3), "| test acc:", round(t.score(X_test, y_test), 3))
print("Full tree depth:", tree_full.get_depth())

Full-depth tree | train acc: 1.0 | test acc: 0.672
Depth-3 tree | train acc: 0.822 | test acc: 0.778
Full tree depth: 15


## Task 3.4

In [6]:
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]
print("RF train acc:", round(rf.score(X_train, y_train), 3))
print("RF test acc:", round(accuracy_score(y_test, rf_pred), 3), "recall:", round(recall_score(y_test, rf_pred), 3),
      "ROC-AUC:", round(roc_auc_score(y_test, rf_prob), 3))
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp.head(3).round(3))

t3_pred = tree3.predict(X_test)
t3_prob = tree3.predict_proba(X_test)[:, 1]
table = pd.DataFrame({
    "model": ["Logistic regression", "Depth-3 tree", "Random forest"],
    "accuracy": [lr_acc, accuracy_score(y_test, t3_pred), accuracy_score(y_test, rf_pred)],
    "recall": [lr_rec, recall_score(y_test, t3_pred), recall_score(y_test, rf_pred)],
    "roc_auc": [lr_auc, roc_auc_score(y_test, t3_prob), roc_auc_score(y_test, rf_prob)]}).round(3)
print(table)

RF train acc: 1.0
RF test acc: 0.747 recall: 0.306 ROC-AUC: 0.706
lead_days      0.220
age            0.197
distance_km    0.195
dtype: float64
                 model  accuracy  recall  roc_auc
0  Logistic regression     0.788   0.347    0.789
1         Depth-3 tree     0.778   0.224    0.700
2        Random forest     0.747   0.306    0.706


## Task 3.5

In [7]:
# (a) tune the forest on the training set
grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    {"n_estimators": [100, 200], "max_depth": [3, 5, 8]}, cv=5, scoring="roc_auc")
grid.fit(X_train, y_train)
print("best_params_:", grid.best_params_, "| best_score_ (CV ROC-AUC):", round(grid.best_score_, 3))

# (b) choose the model by 5-fold CV ROC-AUC on the training set
lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
models = {"Logistic regression": lr_pipe,
          "Depth-3 tree": DecisionTreeClassifier(max_depth=3, random_state=42),
          "Random forest": RandomForestClassifier(n_estimators=100, random_state=42)}
cv_auc = {name: cross_val_score(m, X_train, y_train, cv=5, scoring="roc_auc").mean() for name, m in models.items()}
cv_auc["Tuned forest"] = grid.best_score_
models["Tuned forest"] = grid.best_estimator_
for name, s in cv_auc.items():
    print(f"{name:22s} CV ROC-AUC {s:.3f}")
winner = max(cv_auc, key=cv_auc.get)
final = models[winner]
print("Winner:", winner)

# (c) choose the threshold on out-of-fold training probabilities
oof = cross_val_predict(final, X_train, y_train, cv=5, method="predict_proba")[:, 1]
best_thr, best_f1 = None, -1
for thr in [0.5, 0.35]:
    flag = (oof >= thr).astype(int)
    f1 = f1_score(y_train, flag)
    print(f"threshold {thr}: precision {precision_score(y_train, flag):.3f} recall {recall_score(y_train, flag):.3f} F1 {f1:.3f} flagged {flag.sum()}")
    if f1 > best_f1:
        best_thr, best_f1 = thr, f1
print("Chosen threshold:", best_thr)

# (d) fit on the training set and evaluate once on the test set
final.fit(X_train, y_train)
probs = final.predict_proba(X_test)[:, 1]
flags = (probs >= best_thr).astype(int)
print("TEST at", best_thr, "| precision:", round(precision_score(y_test, flags), 3), "recall:", round(recall_score(y_test, flags), 3),
      "flagged:", flags.sum(), "ROC-AUC:", round(roc_auc_score(y_test, probs), 3))

# (e) save the test-set scores
out = pd.DataFrame({"appointment_id": df.loc[X_test.index, "appointment_id"].values, "no_show": y_test.values,
                    "risk_prob": probs, "flag": flags})
out.to_csv("test_risk_scores.csv", index=False)
print("test_risk_scores.csv rows:", len(out), "| flag = 1:", out["flag"].sum())

best_params_: {'max_depth': 3, 'n_estimators': 200} | best_score_ (CV ROC-AUC): 0.767
Logistic regression    CV ROC-AUC 0.774
Depth-3 tree           CV ROC-AUC 0.678
Random forest          CV ROC-AUC 0.747
Tuned forest           CV ROC-AUC 0.767
Winner: Logistic regression
threshold 0.5: precision 0.575 recall 0.249 F1 0.347 flagged 73
threshold 0.35: precision 0.513 recall 0.467 F1 0.489 flagged 154
Chosen threshold: 0.35
TEST at 0.35 | precision: 0.571 recall: 0.571 flagged: 49 ROC-AUC: 0.789
test_risk_scores.csv rows: 198 | flag = 1: 49
